# 04 - Organize Persona Dataset

Merges the persona and assistant judge output (notebook 02) with the roleplay generator output
(notebook 03) into one unified dataset, `results/persona_dataset.json`.

Merges:
- **`results/CADFEB_results/CADFEB_assistant.jsonl`** — rows where `is_safe_for_assistant ==
  True` become `role="assistant"`, `source="assistant"`.
- **`results/CADFEB_results/CADFEB_persona.jsonl`** — every successful row becomes
  `role=<persona label>` (`sycophantic` / `malicious-manipulative` / `evil`), except
  `persona == "other"` which becomes `role="other (toxic)"` with `other_role` set to the judge's
  `other_label`. All get `source="toxic"`.
- **`results/roleplay_results/*.jsonl`** — every successful row becomes `role=<persona>`
  (`pirate` / `poet`), `source="roleplay"`.

Output schema per row: `post_id, category, category_detail, role, other_role, content, source`.

Writes to a scratch path and diffs against the tracked `results/persona_dataset.json` rather
than overwriting it directly — see the verification step below.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

from src import config

## 1. Load the two upstream sources

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def successful(rows: list[dict]) -> list[dict]:
    return [r for r in rows if "error" not in r]


assistant_rows = successful(load_jsonl(config.CADFEB_RESULTS_DIR / f"{config.CADFEB_ASSISTANT_JUDGE_NAME}.jsonl"))
persona_rows = successful(load_jsonl(config.CADFEB_RESULTS_DIR / f"{config.CADFEB_PERSONA_JUDGE_NAME}.jsonl"))

roleplay_rows = []
for path in sorted(config.ROLEPLAY_RESULTS_DIR.glob("*.jsonl")):
    roleplay_rows.extend(successful(load_jsonl(path)))

print(f"assistant judge: {len(assistant_rows)} successful rows")
print(f"persona judge: {len(persona_rows)} successful rows")
print(f"roleplay: {len(roleplay_rows)} successful rows")

assistant judge: 1140 successful rows
persona judge: 1193 successful rows
roleplay: 600 successful rows


## 2. Merge into the unified persona dataset schema

In [3]:
compiled = []

for row in assistant_rows:
    if not row.get("is_safe_for_assistant"):
        continue
    compiled.append(
        {
            "post_id": row["post_id"],
            "category": row["category"],
            "category_detail": config.CATEGORY_LABELS[row["category"]],
            "role": "assistant",
            "other_role": None,
            "content": row["content"],
            "source": "assistant",
        }
    )

for row in persona_rows:
    persona = row["persona"]
    role = "other (toxic)" if persona == "other" else persona
    compiled.append(
        {
            "post_id": row["post_id"],
            "category": row["category"],
            "category_detail": config.CATEGORY_LABELS[row["category"]],
            "role": role,
            "other_role": row.get("other_label") if persona == "other" else None,
            "content": row["content"],
            "source": "toxic",
        }
    )

for row in roleplay_rows:
    compiled.append(
        {
            "post_id": row["post_id"],
            "category": row["category"],
            "category_detail": config.CATEGORY_LABELS[row["category"]],
            "role": row["persona"],
            "other_role": None,
            "content": row["content"],
            "source": "roleplay",
        }
    )

print(f"{len(compiled)} compiled rows")

2755 compiled rows


## 3. Verify against `results/persona_dataset.json`

Compares the reconstruction row-for-row (by `post_id`) against the tracked file rather than
overwriting it. The underlying judge/roleplay stores are append-only and retry
previously-errored rows on every run, so a handful of `post_id`s can become newly available
after the tracked file was last built — these show up as "extra in reconstruction" rows, not
mismatches, and are not written back.

In [4]:
scratch_path = config.RESULTS_DIR / "persona_dataset.reconstructed.json"
with open(scratch_path, "w", encoding="utf-8") as f:
    json.dump(compiled, f, ensure_ascii=False)

with open(config.RESULTS_DIR / "persona_dataset.json", "r", encoding="utf-8") as f:
    existing = json.load(f)

existing_by_id = {r["post_id"]: r for r in existing}
compiled_by_id = {r["post_id"]: r for r in compiled}

missing_from_compiled = set(existing_by_id) - set(compiled_by_id)
extra_in_compiled = set(compiled_by_id) - set(existing_by_id)
mismatched = [pid for pid in (set(existing_by_id) & set(compiled_by_id)) if existing_by_id[pid] != compiled_by_id[pid]]

print(f"existing: {len(existing)} rows, reconstructed: {len(compiled)} rows")
print(f"missing from reconstruction: {len(missing_from_compiled)}")
print(f"extra in reconstruction: {len(extra_in_compiled)}")
print(f"field mismatches: {len(mismatched)}")

if not missing_from_compiled and not extra_in_compiled and not mismatched:
    print("\nExact match -- reconstruction reproduces results/persona_dataset.json byte-for-byte in content.")
else:
    for pid in list(mismatched)[:5]:
        print(f"\n[{pid}] existing: {existing_by_id[pid]}")
        print(f"[{pid}] built:    {compiled_by_id[pid]}")

scratch_path.unlink()

existing: 2739 rows, reconstructed: 2755 rows
missing from reconstruction: 0
extra in reconstruction: 16
field mismatches: 0


## 4. Summary: role counts

In [5]:
from collections import Counter

print("role counts:")
for role, n in Counter(r["role"] for r in compiled).most_common():
    print(f"  {role}: {n}")

print("\nsource counts:")
for source, n in Counter(r["source"] for r in compiled).most_common():
    print(f"  {source}: {n}")

role counts:
  assistant: 962
  malicious-manipulative: 527
  other (toxic): 356
  pirate: 300
  poet: 300
  evil: 261
  sycophantic: 49

source counts:
  toxic: 1193
  assistant: 962
  roleplay: 600
